In [11]:
import time
import requests
import pandas as pd
import yfinance as yf

In [6]:
def fetch_key_statistics(api_key: str, lang='kr', page_size=100):
    rows, start = [], 1
    while True:
        end = start + page_size - 1
        url = f"https://ecos.bok.or.kr/api/KeyStatisticList/{api_key}/json/{lang}/{start}/{end}"
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        j = r.json()

        # 공통 오류 처리 (RESULT가 있고 CODE가 정상코드가 아닐 때)
        if isinstance(j, dict) and 'RESULT' in j:
            code = j['RESULT'].get('CODE')
            if code and code not in ('0000', 'INFO-000', 'INFO-1000'):
                msg = j['RESULT'].get('MESSAGE')
                raise RuntimeError(f"ECOS API 오류: {code} - {msg}")

        block = j.get('KeyStatisticList')

        # ① dict 형식: {"KeyStatisticList":{"list_total_count":..., "row":[{...}, ...]}}
        items = []
        if isinstance(block, dict):
            items = block.get('row', [])
        # ② list 형식: {"KeyStatisticList":[{"list_total_count":...}, {"CLASS_NAME":...}, ...]}
        elif isinstance(block, list):
            items = [x for x in block if isinstance(x, dict) and 'KEYSTAT_NAME' in x]

        if not items:
            break

        rows.extend(items)

        # 마지막 페이지면 종료
        if len(items) < page_size:
            break

        start = end + 1
        time.sleep(0.2)  # 과호출(602) 방지

    df = pd.DataFrame(rows)

    # 컬럼이 있으면 정리, 없으면 그대로 반환
    wanted = ['CLASS_NAME', 'KEYSTAT_NAME', 'DATA_VALUE', 'CYCLE', 'UNIT_NAME']
    have = [c for c in wanted if c in df.columns]
    if have:
        df = df[have]
        # 타입 다듬기
        if 'DATA_VALUE' in df.columns:
            df['DATA_VALUE'] = pd.to_numeric(df['DATA_VALUE'], errors='coerce')
    return df

In [ ]:
ecos = fetch_key_statistics("BIWA3PF39IWWN5CS5J6R")
print(ecos.head())

In [10]:
ecos.to_csv("../data/ecos_key_statistics.csv", index=False)

In [13]:
TICKERS = {
    "KOSPI": "^KS11",
    "KOSDAQ": "^KQ11",
    "S&P 500": "^GSPC",
    "NASDAQ": "^IXIC",
    "Brent": "BZ=F",
    "WTI": "CL=F",
    "Copper": "HG=F",
    "Gold": "GC=F",
    "Lithium (ETF proxy)": "LIT",
}

def fetch_yf_quotes(ticker_map: dict) -> pd.DataFrame:
    rows = []
    for name, symbol in ticker_map.items():
        t = yf.Ticker(symbol)

        # 최신가(정규장/연장장 포함)와 전일 종가
        info = t.fast_info  # 빠름: last_price, previous_close, currency, timezone 등
        last = getattr(info, "last_price", None)
        prev = getattr(info, "previous_close", None)
        curr = getattr(info, "currency", None)
        tz = getattr(info, "timezone", None)

        # 퍼센트/절대변화
        chg = None if (last is None or prev in (None, 0)) else last - prev
        chg_pct = None if (last is None or prev in (None, 0)) else (last/prev - 1) * 100

        # 타임스탬프(가능하면)
        # 최근 1일 데이터에서 마지막 시점 추정
        try:
            hist = t.history(period="1d", interval="1m")
            ts = hist.index[-1].to_pydatetime() if len(hist) else None
        except Exception:
            ts = None

        rows.append({
            "name": name,
            "symbol": symbol,
            "last": last,
            "previous_close": prev,
            "change": chg,
            "change_pct": chg_pct,
            "currency": curr,
            "timezone": tz,
            "timestamp": ts
        })

    df = pd.DataFrame(rows)
    # 보기 좋게 정렬
    order = ["name","symbol","last","change","change_pct","previous_close","currency","timestamp","timezone"]
    df = df[order]
    return df

시계열도 필요할라나

In [18]:
def fetch_yf_history(ticker_map: dict, period="5y", interval="1d"):
    """각 티커의 과거 시계열 데이터를 모두 가져와 병합"""
    data = {}
    for name, symbol in ticker_map.items():
        try:
            df = yf.download(symbol, period=period, interval=interval, progress=False)
            df = df[["Open", "High", "Low", "Close","Volume"]]
            df.columns = [f"{name}_{col}" for col in df.columns]
            data[name] = df
        except Exception as e:
            print(f"{name} ({symbol}) 다운로드 실패: {e}")

    # 날짜 기준 병합
    combined = pd.concat(data.values(), axis=1, join="outer")
    combined.index.name = "Date"
    return combined


hist = fetch_yf_history(TICKERS, period="3y", interval="1d")
print(hist.tail())
hist.to_csv("../data/market_history_3y.csv")

C:\Users\SKAX\AppData\Local\Temp\ipykernel_19876\1512592179.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, period=period, interval=interval, progress=False)
C:\Users\SKAX\AppData\Local\Temp\ipykernel_19876\1512592179.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, period=period, interval=interval, progress=False)
C:\Users\SKAX\AppData\Local\Temp\ipykernel_19876\1512592179.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, period=period, interval=interval, progress=False)
C:\Users\SKAX\AppData\Local\Temp\ipykernel_19876\1512592179.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, period=period, interval=interval, progress=False)
C:\Users\SKAX\AppData\Local\Temp\ipykernel_19876\1512592179.py:6: FutureWarning: YF.download() has changed argument 

            KOSPI_('Open', '^KS11')  KOSPI_('High', '^KS11')  \
Date                                                           
2025-10-30              4105.950195              4146.720215   
2025-10-31              4083.250000              4122.089844   
2025-11-03              4123.359863              4221.919922   
2025-11-04                      NaN                      NaN   
2025-11-05              4055.469971              4055.469971   

            KOSPI_('Low', '^KS11')  KOSPI_('Close', '^KS11')  \
Date                                                           
2025-10-30             4070.790039               4086.889893   
2025-10-31             4059.739990               4107.500000   
2025-11-03             4123.359863               4221.870117   
2025-11-04                     NaN                       NaN   
2025-11-05             3867.810059               3912.209961   

            KOSPI_('Volume', '^KS11')  KOSDAQ_('Open', '^KQ11')  \
Date                               

In [14]:
quotes = fetch_yf_quotes(TICKERS)
print(quotes)
quotes.to_csv("../data/yf_quotes.csv", index=False)

                  name symbol          last      change  change_pct  \
0                KOSPI  ^KS11   3874.590088 -246.369873   -5.978458   
1               KOSDAQ  ^KQ11    872.119995  -53.970032   -5.827731   
2              S&P 500  ^GSPC   6771.549805  -80.420195   -1.173680   
3               NASDAQ  ^IXIC  23348.636719 -486.086281   -2.039404   
4                Brent   BZ=F     63.669998   -1.060005   -1.637579   
5                  WTI   CL=F     60.139999   -0.730000   -1.199276   
6               Copper   HG=F      4.930500   -0.080500   -1.606468   
7                 Gold   GC=F   3948.899902  -46.400146   -1.161368   
8  Lithium (ETF proxy)    LIT     59.250000   -2.580000   -4.172732   

   previous_close currency                  timestamp          timezone  
0     4120.959961      KRW  2025-11-05 10:39:00+09:00        Asia/Seoul  
1      926.090027      KRW  2025-11-05 10:39:00+09:00        Asia/Seoul  
2     6851.970000      USD  2025-11-04 15:59:00-05:00  America/New_

In [15]:
from fredapi import Fred
fred = Fred(api_key="fb8a5fd4f9e0d2bdd044c6b60ed0f2c0 ")

# 예: 미국 기준금리 (Federal Funds Rate)
fred_df = fred.get_series('FEDFUNDS')
print(fred_df.tail())
fred_df.to_csv("../data/fred.csv", index=False)

2025-06-01    4.33
2025-07-01    4.33
2025-08-01    4.33
2025-09-01    4.22
2025-10-01    4.09
dtype: float64
